# Aula 1 · Notebook 07 — Sua primeira análise (Bloco 3)

Objetivo: **responder 3 perguntas reais** sobre um dataset de e-commerce brasileiro, e depois inventar a sua quarta pergunta.

Este é o notebook **principal do Bloco 3** da aula.

---

## O dataset: Olist

Dataset público da **Olist**, empresa que conecta pequenos lojistas a marketplaces. ~96 mil pedidos entregues entre 2016 e 2018.

| Coluna | O que é |
|--------|---------|
| `order_id` | ID do pedido |
| `customer_state` | UF do cliente |
| `customer_city` | Cidade do cliente |
| `purchase_date` | Data da compra |
| `price`, `freight_value` | Preço e frete |
| `product_category_en` | Categoria do produto |
| `delivery_days` | Dias até a entrega |
| `delivered_late` | Atrasou? (bool) |
| `review_score` | Estrelas (1-5) |
| `review_comment_message` | Comentário em PT-BR |
| ... (32 colunas no total) |

Tempo estimado: 50 minutos.

## Passo 1 — Baixar o dataset

In [1]:
import urllib.request
import pathlib

URL = "https://github.com/fenakamuta/poliusppro-data-engineering/releases/download/aula1-olist-2026-v1/olist.parquet"
PATH = pathlib.Path("data/olist.parquet")
PATH.parent.mkdir(exist_ok=True)

if not PATH.exists():
    print("Baixando olist.parquet (~8 MB)...")
    urllib.request.urlretrieve(URL, PATH)
    print(f"✅ Baixado: {PATH.stat().st_size / 1e6:.1f} MB")
else:
    print(f"Já está baixado: {PATH.stat().st_size / 1e6:.1f} MB")

Baixando olist.parquet (~8 MB)...
✅ Baixado: 8.4 MB


## Passo 2 — Dar uma espiada

In [2]:
import duckdb

duckdb.sql("SELECT * FROM 'data/olist.parquet' LIMIT 5").df()

,order_id,customer_id,customer_state,customer_city,order_status,purchase_timestamp,purchase_date,estimated_delivery_date,delivered_customer_date,delivery_days,...,seller_id,seller_state,seller_city,payment_type,payment_installments,payment_value,review_score,review_positivo,review_comment_title,review_comment_message
0,00e7ee1b050b8499577073aeb2a297a1,06b8999e2fba1a1fbc88172c00ba8bc7,SP,franca,delivered,2017-05-16 15:05:35,2017-05-16,2017-06-05,2017-05-25,9,...,7c67e1448b00f6e969d365cea6b010ab,SP,itaquaquecetuba,credit_card,2,146.87,4,True,None,None
1,29150127e6685892b6eab3eec79f59c7,18955e83d337fd6b2def6b18a428ac77,SP,sao bernardo do campo,delivered,2018-01-12 20:48:24,2018-01-12,2018-02-06,2018-01-29,17,...,b8bc237ba3788b23da09c0f1f3a3288c,SC,itajai,credit_card,8,335.48,5,True,None,None
2,b2059ed67ce144a36e2aa97d2c9e9ad2,4e7b3e00288586ebd08712fdd0374a03,SP,sao paulo,delivered,2018-05-19 16:07:45,2018-05-19,2018-06-13,2018-06-14,26,...,7c67e1448b00f6e969d365cea6b010ab,SP,itaquaquecetuba,credit_card,7,157.73,5,True,None,None
3,951670f92359f4fe4a63112aa7306eba,b2b6027bc5c5109e529d4dc6358b12c3,SP,mogi das cruzes,delivered,2018-03-13 16:06:38,2018-03-13,2018-04-10,2018-03-28,15,...,7c67e1448b00f6e969d365cea6b010ab,SP,itaquaquecetuba,credit_card,1,173.30,5,True,None,None
4,6b7d50bd145f6fc7f33cebabd7e49d0f,4f2d8ab171c80ec8364f7c12e35b23ad,SP,campinas,delivered,2018-07-29 09:51:30,2018-07-29,2018-08-15,2018-08-09,11,...,4a3ca9315b744ce9f8e9374361493884,SP,ibitinga,credit_card,8,252.25,5,True,a melhor nota,O baratheon è esxelente Amo adoro o baratheon


In [3]:
# Quantas linhas e colunas?
duckdb.sql("""
    SELECT COUNT(*) AS pedidos,
           COUNT(DISTINCT customer_state) AS estados,
           COUNT(DISTINCT product_category_en) AS categorias
    FROM 'data/olist.parquet'
""").df()

,pedidos,estados,categorias
0,96478,27,71


---

## Pergunta 1 — Qual estado tem o melhor review médio?

**Hipótese:** estados maiores (mais oferta, mais competição) talvez tenham scores melhores.

In [4]:
duckdb.sql("""
    SELECT customer_state,
           COUNT(*) AS pedidos,
           ROUND(AVG(review_score), 2) AS score_medio
    FROM 'data/olist.parquet'
    GROUP BY customer_state
    ORDER BY score_medio DESC
    LIMIT 10
""").df()

,customer_state,pedidos,score_medio
0,SP,40501,4.25
1,AP,67,4.24
2,AM,145,4.24
3,PR,4923,4.24
4,MG,11354,4.19
5,RS,5345,4.19
6,RO,243,4.17
7,MS,701,4.16
8,MT,886,4.15
9,RN,474,4.15


### Discussão

Repare como o `score_medio` quase não varia entre estados — todos próximos de 4. A **média esconde nuance**.

**Para pensar:** se o objetivo é "melhorar a satisfação", olhar a média não ajuda. Precisaríamos olhar a **proporção de notas baixas** por estado.

---

## Pergunta 2 — Quanto a entrega atrasa, e quanto isso impacta?

In [5]:
duckdb.sql("""
    SELECT delivered_late,
           COUNT(*) AS pedidos,
           ROUND(AVG(delivery_days), 1) AS dias_medio,
           ROUND(AVG(review_score), 2) AS score_medio
    FROM 'data/olist.parquet'
    WHERE delivery_days IS NOT NULL
    GROUP BY delivered_late
""").df()

,delivered_late,pedidos,dias_medio,score_medio
0,False,89936,10.9,4.29
1,True,6534,33.9,2.27


### Discussão

Pedidos no prazo: ~11 dias, score ~4,3.
Pedidos atrasados: ~34 dias, score ~2,4.

**Atraso vira review ruim.** Esse é o tipo de insight que justifica investir em logística. Vamos voltar nele na Aula 4 quando treinarmos um modelo que prevê risco de atraso.

---

## Pergunta 3 — Quais categorias têm os reviews mais negativos?

Para o time de operações: "onde investigar primeiro".

In [6]:
duckdb.sql("""
    SELECT product_category_en,
           COUNT(*) AS pedidos,
           ROUND(AVG(review_score), 2) AS score_medio
    FROM 'data/olist.parquet'
    WHERE product_category_en IS NOT NULL
    GROUP BY product_category_en
    HAVING pedidos > 500
    ORDER BY score_medio ASC
    LIMIT 10
""").df()

,product_category_en,pedidos,score_medio
0,office_furniture,1246,3.65
1,bed_bath_table,9167,4.01
2,telephony,4076,4.06
3,furniture_decor,6213,4.07
4,computers_accessories,6501,4.08
5,baby,2763,4.12
6,electronics,2507,4.13
7,watches_gifts,5472,4.13
8,construction_tools_construction,728,4.13
9,auto,3793,4.15


### Discussão

Categorias com mais reviews ruins são candidatas a investigação: produto com defeito? Embalagem ruim? Expectativa errada na descrição?

**Engenheiro de dados entrega "onde olhar primeiro"; o time de produto investiga.**

---

## Pergunta 4 — Sua vez

Pense numa pergunta que **você teria interesse** em responder. Algumas ideias:

- Qual o **valor médio de frete** por categoria?
- Quem demora mais para entregar: vendedores do **mesmo estado** do cliente, ou de outro estado?
- Existe **mês do ano** em que os reviews são piores?
- Quanto custa em média uma **`bed_bath_table`** comparada a **`computers_accessories`**?

Escreva sua query abaixo (e celebre quando funcionar):

In [3]:
df = duckdb.sql("""
    SELECT *
    FROM 'data/olist.parquet'
""").df()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 96478 entries, 0 to 96477
Data columns (total 32 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   order_id                 96478 non-null  object        
 1   customer_id              96478 non-null  object        
 2   customer_state           96478 non-null  object        
 3   customer_city            96478 non-null  object        
 4   order_status             96478 non-null  object        
 5   purchase_timestamp       96478 non-null  datetime64[us]
 6   purchase_date            96478 non-null  datetime64[us]
 7   estimated_delivery_date  96478 non-null  datetime64[us]
 8   delivered_customer_date  96470 non-null  datetime64[us]
 9   delivery_days            96470 non-null  Int64         
 10  estimated_delivery_days  96478 non-null  int64         
 11  delivered_late           96470 non-null  boolean       
 12  product_id               96478 n

In [1]:
# - Qual o **valor médio de frete** por categoria?
import duckdb

duckdb.sql("""
    SELECT
        product_category,
        AVG(freight_value) AS frete_medio
    FROM 'data/olist.parquet'
    GROUP BY product_category
    ORDER BY frete_medio
""").df()

,product_category,frete_medio
0,fashion_roupa_infanto_juvenil,11.245714
1,livros_importados,12.482653
2,fashion_roupa_feminina,13.704167
3,casa_conforto_2,14.240455
4,alimentos,14.381701
...,...,...
69,moveis_cozinha_area_de_servico_jantar_e_jardim,42.494142
70,moveis_colchao_e_estofado,42.739730
71,moveis_quarto,42.807865
72,eletrodomesticos_2,44.849381


In [18]:
# - Quem demora mais para entregar: vendedores do **mesmo estado** do cliente, ou de outro estado?
duckdb.sql("""
    SELECT
        CASE
            WHEN customer_state = seller_state THEN 'mesmo_estado'
            ELSE 'outro_estado'
        END AS state_seller_customer,
        ROUND(AVG(delivery_days), 1) as frete_medio
    FROM 'data/olist.parquet'
    GROUP BY state_seller_customer
    ORDER BY state_seller_customer
""").df()

,state_seller_customer,frete_medio
0,mesmo_estado,7.9
1,outro_estado,15.1


In [21]:
# - Existe **mês do ano** em que os reviews são piores?
duckdb.sql("""
    SELECT
        EXTRACT(MONTH FROM purchase_date) as mes,
        COUNT(*) AS pedidos,
        ROUND(AVG(review_score), 1) as score_medio
    FROM 'data/olist.parquet'
    GROUP BY mes
    ORDER BY mes
""").df()

,mes,pedidos,score_medio
0,1,7819,4.1
1,2,8208,3.9
2,3,9549,3.9
3,4,9101,4.2
4,5,10295,4.2
5,6,9234,4.3
6,7,10031,4.3
7,8,10544,4.3
8,9,4151,4.3
9,10,4743,4.2


In [24]:
# - Quanto custa em média uma **`bed_bath_table`** comparada a **`computers_accessories`**?
duckdb.sql("""
    SELECT
        product_category_en AS categoria,
        COUNT(*) AS pedidos,
        ROUND(AVG(price), 1) as preco_medio
    FROM 'data/olist.parquet'
    WHERE product_category_en IN ('bed_bath_table','computers_accessories')
    GROUP BY categoria
    ORDER BY categoria
""").df()

,categoria,pedidos,preco_medio
0,bed_bath_table,9167,96.4
1,computers_accessories,6501,116.7


---

## Passo final — Salvar e versionar no GitHub

Sua análise está pronta? Vamos salvar e mandar pro GitHub.

In [25]:
import datetime

resumo = duckdb.sql("""
    SELECT customer_state,
           COUNT(*) AS pedidos,
           ROUND(AVG(review_score), 2) AS score,
           ROUND(AVG(delivery_days), 1) AS dias_medio
    FROM 'data/olist.parquet'
    GROUP BY customer_state
    ORDER BY pedidos DESC
""").df()

caminho = "data/minha_analise_olist.csv"
resumo.to_csv(caminho, index=False)
print(f"✅ Análise salva em {caminho}")
print(f"   Gerada em {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}")

✅ Análise salva em data/minha_analise_olist.csv
   Gerada em 2026-06-21 22:10


### Comitar no Git

No terminal (Cursor → Ctrl/Cmd + ` para abrir):

```bash
git init                        # se ainda não tiver
git add data/minha_analise_olist.csv 07-bloco-3-olist-analise.ipynb
git commit -m "minha primeira análise de dados real"
```

Para subir no GitHub, crie um repositório vazio na sua conta e:

```bash
git remote add origin https://github.com/jlfcar2/olist-analise.git
git branch -M main
git push -u origin main
```

Pronto: você tem um portfólio começando.

---

## Resumo da aula

| Você sabia? | Agora sabe |
|-------------|-----------|
| Que dado vinha em CSV | Que vem em CSV, JSON e Parquet — e cada um serve pra algo |
| Que SQL precisava de servidor | Que DuckDB roda SQL direto em arquivo |
| Que análise era em planilha | Que análise pode ser em código, versionada no Git |

---

🎉 **Parabéns. Sua primeira análise real está pronta.**